In [1]:
!pip install -q openai

In [3]:
from google.colab import userdata
from openai import OpenAI
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Sure, here's one for you:

**Why don’t scientists trust atoms?**

*Because they make up everything!* 😄


In [4]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We’d love to make an egg‑less cake for you! However, we need at least 24 hours’ notice, so today evening isn’t possible. Please place your order for tomorrow and we’ll bake it fresh!
CompletionUsage(completion_tokens=177, prompt_tokens=183, total_tokens=360, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=122, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.077000157, prompt_time=0.008760167, completion_time=0.371690125, total_time=0.380450292)


In [5]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

In [12]:
#MENU["donut"]
MENU.get("donut", "donuts not available")


'donuts not available'

In [13]:
order = {"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": True}

In [14]:
order["qty"] * MENU[order["item"]] # 1000

1000

In [15]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [16]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [17]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [18]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [19]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [20]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: Hi
Bot: Hello! 👋 Welcome to Sweet Crumbs Home Bakery. How can I help you today—perhaps with a cake or cupcakes? Let me know your preferences!
You: Cupcake
Bot: Great! Our cupcakes are ₹80 each and we also have an egg‑less version. How many would you like, and would you prefer the regular or egg‑less ones? Let me know!
You: Eggless
Bot: Sure! Egg‑less cupcakes are ₹80 each. How many would you like to order? (Please note we need a 24‑hour notice.) Let me know the quantity!
You: i want to order 8 cupcakes
Bot: Got it! 8 egg‑less cupcakes will be ₹640 total (24‑hour notice required). May I have your name to finalize the order?
You: I am sahithi
Bot: Thanks, Sahithi! Your order: 8 egg‑less cupcakes – ₹640 total (24‑hour notice noted). Please confirm, and we’ll hold it for you. Let me know if you’d like anything else!
You: Do you have red velvet cake?
Bot: I'll check with the owner about a red‑velvet cake. Meanwhile, would you like anything else from our menu?
You: Give me the menu once

In [21]:
history

[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'Hi'},
 {'role': 'assistant',
  'content': 'Hello! 👋 Welcome to Sweet Crumbs Home Bakery. How can I help you today—perhaps with a cake or cupcakes? Let me know your preferences!'},
 {'role': 'user', 'content': 'Cupcake'},
 {'role': 'assistant',
  'content': 'Great! Our cupcakes are ₹80 each and we also have an egg‑less version. How many would you like, and would you prefer the regular or egg‑less ones? Let me know!'},
 {'role': 'user', 'content': 'Eggless